# Clinical RAG Platform — Colab Backend

Run the cells in order. Add `COLAB_API_KEY` (at least 16 characters), `ASPNET_BACKEND_URL` (the public HTTPS ASP.NET origin) and `COLAB_REGISTRATION_SECRET` in Colab Secrets (key icon). The registration secret must match backend `Colab__RegistrationSecret`. Optionally add `MONGODB_URI` for legacy retrieval only. Runtime: GPU (T4 or better).

This notebook starts:
1. The model server (Qwen3-4B-Instruct + PubMedBERT + BGE reranker) on `127.0.0.1:8000`
2. A public Cloudflare tunnel on port `8000`, registered automatically in ASP.NET's protected database setting, with 30-second heartbeats.

Do not include `.env` files, appsettings secrets, `key-ring`, patient data or uploads in the application zip. Settings manual override blocks notebook registration; release it and rerun the tunnel cell to re-register.

In [ ]:
# Cell 1: Install the tunnel binary (Python packages are installed from pinned files after upload)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

In [ ]:
# Cell 2: Upload the app zip
from google.colab import files
name = next(iter(files.upload()))   # select clinical-rag.zip
!rm -rf clinical-rag && mkdir clinical-rag && unzip -qo "{name}" -d clinical-rag
!pip -q install -r clinical-rag/requirements.txt -r clinical-rag/requirements-colab.txt

In [ ]:
# Cell 3: Start the secured model server on 127.0.0.1:8000
# This loads Qwen3-4B-Instruct (vLLM) + PubMedBERT + BGE reranker. Takes ~5 min first time.
import os, sys, threading, uvicorn
from google.colab import userdata
os.environ['COLAB_API_KEY'] = userdata.get('COLAB_API_KEY')
try:
    os.environ['COLAB_MONGODB_URI'] = userdata.get('MONGODB_URI')
except Exception:
    pass
sys.path.insert(0, 'clinical-rag')
from colab.server import app
threading.Thread(target=lambda: uvicorn.run(app, host='127.0.0.1', port=8000, log_level='warning', access_log=False), daemon=True).start()

In [ ]:
# Cell 4: Run offline tests of the deterministic pipeline (plan, verify, rules)
!cd clinical-rag && python -m pytest -q tests

In [ ]:
# Cell 5: Public Cloudflare tunnel (port 8000 matches the uvicorn server)
import os, re, subprocess, threading
from colab.registration import start_registration, wait_for_model_ready
try:
    backend_url = userdata.get('ASPNET_BACKEND_URL')
    registration_secret = userdata.get('COLAB_REGISTRATION_SECRET')
except Exception:
    raise RuntimeError('Add ASPNET_BACKEND_URL and COLAB_REGISTRATION_SECRET in Colab Secrets.') from None
model_status = wait_for_model_ready(os.environ['COLAB_API_KEY'])
if 'registration_handle' in globals():
    registration_handle.stop()
p = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8000'], stderr=subprocess.PIPE, text=True)
for line in p.stderr:
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        tunnel_url = m.group(0)
        break
else:
    raise RuntimeError('Cloudflare tunnel did not start. Restart this cell.')
def drain_tunnel_logs():
    for _ in p.stderr:
        pass
threading.Thread(target=drain_tunnel_logs, daemon=True).start()
registration_handle = start_registration(backend_url, tunnel_url, os.environ['COLAB_API_KEY'], registration_secret)
print('Colab registered. Settings will show live status and the last heartbeat.')
print('Model:', model_status.get('model'), '| GPU:', model_status.get('gpu', {}).get('name', 'unknown'))